# Pulsed-SABRE: Analytical model for a three-spin system
In this notebook, we will derive an analytical model for a three-spin system that is subjected to a pulsed polarisation transfer field. We consider a square-modulated field and derive a time-independent representation of the Hamiltonian using rotating frame transformations and average Hamiltonian theory. Finally, we show that the generalised LAC condition applies also for square pulses and we calculate analytical expressions for the off-diagonal drive.

The calculations are made using SymPy.

In [ ]:
from sympy import *

### Define symbols and Pauli matrices
We will define all the necessary symbols and Pauli matrices.

In [ ]:
sx, sz, s0z, s1z = symbols("s_x, sz, s_0z, s_1z", real=True)
wp, t, tp = symbols(r"\omega_p, t, t'", positive=True, real=True)
s1zf = Function("s_1z", real=True)(t)
m = symbols("m", integer=True)

sigma_x = Matrix([
    [0, 1],
    [1, 0]
])

sigma_z = Matrix([
    [1, 0],
    [0, -1]
])

### Write the time-dependent Hamiltonian
The Hamiltonian that describes the coherent spin dynamics that drive the transitions between $\ket{S\alpha}$ and $\ket{T_+\beta}$ (or $\ket{S\beta}$ and $\ket{T_-\alpha}$) can be reduced into a $2 \times 2$ block that contains the states of interest. This Hamiltonian is of form:
$$
\hat{H} = \begin{bmatrix} s_{0z} + s_{1z}(t)  & s_x \\ s_x & -s_{0z} - s_{1z}(t) \end{bmatrix}
$$
The coefficients $s_x$, $s_{0z}$, and $s_{1z}(t)$ contain terms from Zeeman interaction and $J$-coupling and they are presented elsewhere.

In [ ]:
H = sx*sigma_x + (s0z + s1zf)*sigma_z
display(H)

### Transform to the first rotating frame
We want to apply a rotating frame transformation to an interaction frame defined by the time-dependent term $s_{1z}(t)\hat{\sigma}_z$. The unitary operator that achieves this transformation is:
$$
\hat{U}_1 = e^{-i\hat{\sigma}_z \int_{0}^{t} s_{1z}(t')dt'}
$$
The unitary transformation is applied to the Hamiltonian by:
$$
\tilde{H} = \hat{U}_1^{\dag}\hat{H}\hat{U}_1 - i\hat{U}_1^{\dag}\dot{\hat{U}}_1
$$

In [ ]:
U1 = exp(-I*sigma_z*integrate(s1zf.subs(t, tp), (tp, 0, t)))
display(U1)

Hr1 = U1.H * H * U1 - I*U1.H*diff(U1, t)
Hr1 = simplify(Hr1)
display(Hr1)

### Transform to the second rotating frame
We apply another rotating frame transformation where the interaction frame is chosen inspired by the Floquet theory. We define $m$ such that it corresponds to the Floquet state for which $2s_{0z} + m\omega_p \approx 0$. The unitary operator of interest is:
$$
\hat{U}_2 = e^{\frac{i\hat{\sigma}_z}{2} m\omega_p t}
$$
The transformation is applied again by:
$$
\tilde{\tilde{H}} = \hat{U}_2^{\dag}\tilde{H}\hat{U}_2 - i\hat{U}_2^{\dag}\dot{\hat{U}}_2
$$

In [ ]:
U2 = exp(Rational(1/2)*I*sigma_z*m*wp*t)
display(U2)

Hr2 = U2.H * Hr1 * U2 - I*U2.H*diff(U2, t)
Hr2 = simplify(Hr2)
display(Hr2)

### Consider the square wave
The above Hamiltonian is still exact formulation of the $2 \times 2$ representation. However, it is of our interest to derive a time-independent form of the Hamiltonian. We will choose to deal with the following square wave:
$$
s_{1z}(t) = 
\begin{cases} 
    s_{1z} & 0 \leq t < \pi/\omega_p \\
    - s_{1z} & \pi/\omega_p \leq t < 2\pi/\omega_p 
\end{cases}
$$
The Hamiltonian will be split into two parts that we call part A and B.

In [ ]:
Hr2A = simplify(Hr2.subs(
    Integral(s1zf.subs(t, tp), (tp, 0, t)), Integral(s1z, (tp, 0, t))
))
display(Hr2A)

Hr2B = simplify(Hr2.subs(
    Integral(s1zf.subs(t, tp), (tp, 0, t)),
    Integral(s1z, (tp, 0, pi/wp)) + Integral(-s1z, (tp, pi/wp, t))
))
display(Hr2B)

### Apply the average Hamiltonian theory
We use the first-order average from the average Hamiltonian theory, which is simply:
$$
\tilde{\tilde{H}}^{(1)} = \frac{1}{2\pi/\omega_p} \int_{0}^{2\pi/\omega_p} \tilde{\tilde{H}} dt
$$

In [ ]:
Hr2A_eff = simplify(integrate(Hr2A, (t, 0, pi/wp)))
Hr2B_eff = simplify(integrate(Hr2B, (t, pi/wp, 2*pi/wp)))

Hr2_eff = (Hr2A_eff + Hr2B_eff) / (2*pi / wp)
Hr2_eff = simplify(Hr2_eff)
display(Hr2_eff)

### Interpreting the result
We find again that the LAC condition is exactly the same as for cosine pulses:
$$
2s_{0z} + m\omega_p = 0
$$
This is not surprising, as any wave that could be written as a Fourier series can be interpreted using Floquet theory, where the LACs between the Floquet states occur at the exact condition presented here. However, we find that the off-diagonal elements are different.